# Class 11 - Structure And Next Route

最后不是多学几个库，而是把 micrograd 放进下一阶段路线。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

import importlib
import course.checks as course_checks
importlib.reload(course_checks)
qa_check = course_checks.qa_check

## 这节不是列资源，是整理主线

学完 micrograd 后，你不应该只得到“我看过一个项目”。你应该得到一条流程：

```text
Value 标量自动求导
-> Neuron / Layer / MLP
-> loss
-> backward
-> 参数更新
-> 分类边界
-> PyTorch Tensor / optimizer
```

后面所有大模型课程，都是这条链路变大：数据更多、参数更多、张量更高维、训练工程更复杂。

In [ ]:
structure_map = {
    'micrograd/engine.py': 'Value：data、grad、计算图、backward',
    'micrograd/nn.py': 'Neuron / Layer / MLP：参数组织和 forward',
    'course/06_training_loop': 'forward -> loss -> zero_grad -> backward -> update',
    'course/07_classification_boundary': 'score、margin、hinge loss、decision boundary',
    'course/08_pytorch_bridge': '把 Value/backward/update 翻译到 PyTorch',
}

for path, role in structure_map.items():
    print(f'{path:36s} -> {role}')

## 后续路线怎么判断“学会了”

不要用“看完某个 repo”当目标。每一站都要有过关问题。

| 阶段 | 过关问题 |
|---|---|
| PyTorch | 我能不用 micrograd，写出同一个训练循环吗？ |
| D2L | 我能解释线性回归、MLP、优化器为什么能训练吗？ |
| Happy-LLM | 我能画出 Transformer 的数据流吗？ |
| LLMs-from-scratch | 我能从 token ids 一路讲到 logits 和 generate 吗？ |
| nanoGPT | 我能看懂真实训练循环、checkpoint、batch、优化细节吗？ |

这才是路线，不是收藏夹。

In [ ]:
route = [
    {'stage': 'PyTorch', 'action': '重写 06 的训练循环', 'pass_question': 'loss.backward 和 optimizer.step 各自做什么？'},
    {'stage': 'D2L', 'action': '补线性回归、MLP、优化、Attention', 'pass_question': '为什么梯度下降能让 loss 下降？'},
    {'stage': 'Happy-LLM', 'action': '建立 Transformer/LLM 总地图', 'pass_question': 'attention 在输入输出形状上做了什么？'},
    {'stage': 'LLMs-from-scratch', 'action': '手写 GPT 流程', 'pass_question': 'token ids 如何变成下一个 token 的概率？'},
    {'stage': 'nanoGPT', 'action': '读真实训练工程', 'pass_question': 'batch、checkpoint、mixed precision 分别解决什么问题？'},
]

for item in route:
    print(item['stage'])
    print('  action:', item['action'])
    print('  pass  :', item['pass_question'])

## Debug Lab - 路线过载

如果你同时打开 D2L、Happy-LLM、LLMs-from-scratch、nanoGPT、Hugging Face，一定会晕。

调试方法：只问这一周的唯一产出是什么。

```text
坏目标：我要学完 PyTorch。
好目标：我用 PyTorch 重写 micrograd 的一轮训练，并解释每个 grad。
```

课程结束后，你每天学一颗，不是每天收藏一堆。

## 课堂检查：文件职责和下一站路线

先自己填下面的 Predict / Modify，再展开提示或答案。

## Predict - 文件职责

In [ ]:
# 填写说明：
# - 完成 student_structure_probe，用项目模块检查 engine.py 和 nn.py 的职责。
# - 返回顺序固定：engine_has_value、nn_has_neuron、nn_has_mlp。
# - 运行本 cell，看 qa_check 的提示。


def student_structure_probe():
    import micrograd.engine as engine
    import micrograd.nn as nn
    # TODO: return engine_has_value, nn_has_neuron, nn_has_mlp
    pass


qa_check('qa_check_class_11_predict', globals())


<details><summary>Show / Hide 本题提示</summary>

- `engine` 负责自动求导的 `Value`。
- `nn` 负责网络结构，至少要能找到 `Neuron` 和 `MLP`。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_structure_probe():
    import micrograd.engine as engine
    import micrograd.nn as nn
    return hasattr(engine, 'Value'), hasattr(nn, 'Neuron'), hasattr(nn, 'MLP')
```

</details>


## Run - 项目结构存在性检查

In [ ]:
for path in ['micrograd/engine.py', 'micrograd/nn.py', 'demo.ipynb', 'course/08_pytorch_bridge/homework.ipynb']:
    p = ROOT / path
    print(path, 'exists=', p.exists())

## 拆开看 - 后续路线不是资源清单

In [ ]:
route = [
    ('PyTorch', '把 Value/backward/update 翻译成 Tensor/loss.backward/optimizer.step'),
    ('D2L', '补线性回归、MLP、优化、Attention'),
    ('Happy-LLM', '建立 Transformer/LLM 总地图'),
    ('LLMs-from-scratch', '手写 GPT 流程'),
    ('nanoGPT', '看真实训练工程'),
]
for stage, pass_question in route:
    print(stage, '->', pass_question)

## Modify - 给 PyTorch 写过关问题

In [ ]:
# 填写说明：
# - 完成 student_pytorch_gate_demo，用一个小实验作为“PyTorch 入门过关题”。
# - 运行本 cell，看 qa_check 的提示。

def student_pytorch_gate_demo():
    try:
        import torch
    except Exception:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO: 复用 w=2,x=3,y=7 的例子，返回 loss、grad、更新后的 w。
    pass


qa_check('qa_check_class_11_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- 一个真正的过关题应该能跑出 `loss.backward()` 后的梯度，而不是只说“我会 import torch”。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_pytorch_gate_demo():
    import torch
    w = torch.tensor(2.0, requires_grad=True)
    loss = (w * 3 - 7) ** 2
    loss.backward()
    grad = w.grad.item()
    with torch.no_grad():
        w -= 0.1 * w.grad
    return loss.item(), grad, w.item()
```

</details>


## What To Remember

```text
1. engine.py 管自动求导，nn.py 管神经网络结构。
2. micrograd 的主线是 forward -> loss -> backward -> update。
3. PyTorch 不是新魔法，是同一套概念的张量版。
4. 后续学习路线必须用过关问题驱动，不能只列资源。
5. 如果学晕了，回到一个最小训练循环。
```